# Example 7.8.1 / 1: Supplying a Start from Python

Companion notebook for **7.8.1 MIP Starts and Good-Enough Solutions** in the AMPL lab Documentation, section "Supplying a Start from Python". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Supplying a Start from Python

This small model chooses up to two items; the start sets every `x[i]` to 0 with `set_value`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set I := 1..3;                              # three candidate items
var x {I} binary;                           # 1 if the item is chosen
maximize Selected: sum {i in I} x[i];       # number of items chosen
subject to Limit: sum {i in I} x[i] <= 2;   # at most two items

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
# Build a complete all-zero start from the instantiated variable keys.
candidate = {i: 0 for i in ampl.var["x"].to_dict()}
for i, value in candidate.items():
    ampl.var["x"][i].set_value(value)
ampl.option["solver_msg"] = 0
ampl.solve(solver="highs", highs_options="alg:start=1")
print(ampl.solve_result)

HiGHS 1.14.0:   alg:start = 1
solved


### Pure AMPL: model.run

The same start in pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
let {i in I} x[i] := 0;
option solver highs;
option solver_msg 0;
option highs_options 'alg:start=1';
solve;
print solve_result;

HiGHS 1.14.0:   alg:start = 1
solved
